<a href="https://colab.research.google.com/github/14marcos1/curso_colab_2026/blob/main/deepseek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =============================================================================
# ANÁLISE DE MORTALIDADE SIM/DataSUS — ACRE (2010–2023)
# Causas: Doenças do Aparelho Circulatório (I00–I99) e Neoplasias (C00–C97, D00–D48)
# Ambiente: Google Colab
# =============================================================================

# --- 1. INSTALAÇÃO E IMPORTAÇÕES ---
!pip install pyreaddbc dbfread -q

import os
import urllib.request
from pathlib import Path
import pandas as pd
from dbfread import DBF
import pyreaddbc

# --- 2. CONFIGURAÇÕES ---
PASTA_DADOS = Path("dados_sim_ac")
PASTA_DADOS.mkdir(exist_ok=True)

URL_BASE = "ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES"
ANOS = list(range(2010, 2024))  # 2010 a 2023 (14 anos)

# --- 3. FUNÇÕES AUXILIARES ---

def baixar_arquivo(ano):
    """Baixa o arquivo DBC do ano especificado, se ainda não existir."""
    nome_dbc = f"DOAC{ano}.dbc"
    caminho_dbc = PASTA_DADOS / nome_dbc

    if caminho_dbc.exists():
        print(f"  {nome_dbc} já existe. Pulando download.")
        return caminho_dbc

    url = f"{URL_BASE}/{nome_dbc}"
    print(f"  Baixando {nome_dbc} ...")
    try:
        urllib.request.urlretrieve(url, caminho_dbc)
        print(f"  Download concluído: {caminho_dbc}")
        return caminho_dbc
    except Exception as e:
        print(f"  ERRO ao baixar {nome_dbc}: {e}")
        return None

def converter_dbc_para_dbf(caminho_dbc):
    """Converte arquivo DBC para DBF usando pyreaddbc.dbc2dbf()."""
    caminho_dbf = caminho_dbc.with_suffix(".dbf")
    if caminho_dbf.exists():
        print(f"  {caminho_dbf.name} já existe. Pulando conversão.")
        return caminho_dbf
    try:
        pyreaddbc.dbc2dbf(str(caminho_dbc), str(caminho_dbf))
        print(f"  Convertido: {caminho_dbf}")
        return caminho_dbf
    except Exception as e:
        print(f"  ERRO na conversão de {caminho_dbc.name}: {e}")
        return None

def classificar_causa(causa_basica):
    """
    Classifica o código CID-10 da causa básica em:
    'Aparelho Circulatório', 'Neoplasias' ou None.
    Utiliza os 3 primeiros caracteres normalizados.
    """
    if not causa_basica:
        return None

    # Normalização: remover espaços, converter para maiúsculas
    codigo = str(causa_basica).strip().upper()

    # Usar os 3 primeiros caracteres (categoria CID-10)
    if len(codigo) < 3:
        return None
    cat = codigo[:3]

    # Grupo 1 — Doenças do aparelho circulatório: I00 a I99
    if cat.startswith("I") and len(cat) == 3 and cat[1:].isdigit():
        num = int(cat[1:])
        if 0 <= num <= 99:
            return "Aparelho Circulatório"

    # Grupo 2 — Neoplasias: C00–C97 e D00–D48
    if cat.startswith("C") and len(cat) == 3 and cat[1:].isdigit():
        num = int(cat[1:])
        if 0 <= num <= 97:
            return "Neoplasias"

    if cat.startswith("D") and len(cat) == 3 and cat[1:].isdigit():
        num = int(cat[1:])
        if 0 <= num <= 48:
            return "Neoplasias"

    return None

def processar_ano(ano):
    """
    Processa um único ano: baixa, converte, lê DBF registro por registro,
    conta óbitos por grupo. Retorna um dicionário com contagens.
    """
    print(f"\n=== Processando ano {ano} ===")

    # Download
    caminho_dbc = baixar_arquivo(ano)
    if caminho_dbc is None:
        return None

    # Conversão DBC → DBF
    caminho_dbf = converter_dbc_para_dbf(caminho_dbc)
    if caminho_dbf is None:
        return None

    # Leitura do DBF e contagem
    contagem = {"Aparelho Circulatório": 0, "Neoplasias": 0}
    total_registros = 0

    try:
        tabela = DBF(str(caminho_dbf), encoding="iso-8859-1", load=False)
        for registro in tabela:
            total_registros += 1
            causa = registro.get("CAUSABAS", "")
            grupo = classificar_causa(causa)
            if grupo:
                contagem[grupo] += 1

        print(f"  Total de registros lidos: {total_registros}")
        print(f"  Aparelho Circulatório: {contagem['Aparelho Circulatório']}")
        print(f"  Neoplasias: {contagem['Neoplasias']}")

        return contagem

    except Exception as e:
        print(f"  ERRO ao ler DBF {caminho_dbf.name}: {e}")
        return None

# --- 4. PROCESSAMENTO DE TODOS OS ANOS ---

resultados = []
anos_processados = set()
anos_com_erro = []

for ano in ANOS:
    try:
        contagem = processar_ano(ano)
        if contagem is None:
            anos_com_erro.append(ano)
            continue

        resultados.append({"ANO": ano, "GRUPO": "Aparelho Circulatório", "OBITOS": contagem["Aparelho Circulatório"]})
        resultados.append({"ANO": ano, "GRUPO": "Neoplasias", "OBITOS": contagem["Neoplasias"]})
        anos_processados.add(ano)

    except Exception as e:
        print(f"ERRO inesperado no ano {ano}: {e}")
        anos_com_erro.append(ano)

# --- 5. CONSOLIDAÇÃO ---

df = pd.DataFrame(resultados, columns=["ANO", "GRUPO", "OBITOS"])
df = df.sort_values(["ANO", "GRUPO"]).reset_index(drop=True)

print("\n" + "="*60)
print("RESULTADO CONSOLIDADO")
print("="*60)
print(df.to_string(index=False))

# --- 6. SALVAMENTO DO CSV ---
caminho_csv = "mortalidade_ac_consolidado.csv"
df.to_csv(caminho_csv, index=False, encoding="utf-8", sep=",")
print(f"\nArquivo salvo: {caminho_csv}")

# --- 7. VALIDAÇÕES ---

# Verificar se todos os 14 anos foram processados
assert len(anos_processados) == 14, \
    f"ERRO: Apenas {len(anos_processados)} anos processados. Anos com erro: {anos_com_erro}"

# Verificar número de linhas no DataFrame
assert len(df) == 28, f"ERRO: DataFrame contém {len(df)} linhas, esperado 28."

# Verificar se os dois grupos estão presentes em cada ano
for ano in ANOS:
    grupos_ano = set(df[df["ANO"] == ano]["GRUPO"].tolist())
    assert grupos_ano == {"Aparelho Circulatório", "Neoplasias"}, \
        f"ERRO: Ano {ano} não contém ambos os grupos. Grupos encontrados: {grupos_ano}"

# Verificar se o arquivo CSV foi gerado
assert Path(caminho_csv).exists(), f"ERRO: Arquivo {caminho_csv} não foi gerado."

# Verificar se o CSV contém os mesmos 28 registros
df_csv = pd.read_csv(caminho_csv, encoding="utf-8")
assert len(df_csv) == 28, f"ERRO: CSV contém {len(df_csv)} linhas, esperado 28."

# Verificar consistência entre DataFrame e CSV
assert df.shape == df_csv.shape, "ERRO: Dimensões do DataFrame e do CSV não coincidem."
assert list(df.columns) == list(df_csv.columns), "ERRO: Colunas do DataFrame e do CSV não coincidem."

print("\n" + "="*60)
print("TODAS AS VALIDAÇÕES PASSARAM COM SUCESSO")
print("="*60)


=== Processando ano 2010 ===
  DOAC2010.dbc já existe. Pulando download.
  DOAC2010.dbf já existe. Pulando conversão.
  Total de registros lidos: 3009
  Aparelho Circulatório: 606
  Neoplasias: 331

=== Processando ano 2011 ===
  DOAC2011.dbc já existe. Pulando download.
  DOAC2011.dbf já existe. Pulando conversão.
  Total de registros lidos: 3157
  Aparelho Circulatório: 715
  Neoplasias: 380

=== Processando ano 2012 ===
  DOAC2012.dbc já existe. Pulando download.
  DOAC2012.dbf já existe. Pulando conversão.
  Total de registros lidos: 3293
  Aparelho Circulatório: 659
  Neoplasias: 470

=== Processando ano 2013 ===
  DOAC2013.dbc já existe. Pulando download.
  DOAC2013.dbf já existe. Pulando conversão.
  Total de registros lidos: 3318
  Aparelho Circulatório: 698
  Neoplasias: 470

=== Processando ano 2014 ===
  DOAC2014.dbc já existe. Pulando download.
  DOAC2014.dbf já existe. Pulando conversão.
  Total de registros lidos: 3476
  Aparelho Circulatório: 762
  Neoplasias: 496

=== 

In [5]:
# =============================================================================
# VISUALIZAÇÃO INTERATIVA — MORTALIDADE ACRE (SIM/DataSUS)
# Gráfico de linhas: Neoplasias vs Aparelho Circulatório (2010–2023)
# Ambiente: Google Colab
# =============================================================================

# --- 1. INSTALAÇÃO E IMPORTAÇÃO ---
!pip install plotly pandas -q

import os
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# --- 2. LEITURA FLEXÍVEL DO CSV (ANTI-FALHA) ---
ARQUIVO_ALVO = "mortalidade_ac_consolidado.csv"
caminho_csv = None

# 2.1. Busca dinâmica na pasta atual
candidatos = []
for arq in Path(".").glob("*.csv"):
    nome = arq.name.lower()
    if nome.endswith("_consolidado.csv") or nome == ARQUIVO_ALVO.lower():
        candidatos.append(arq)

# 2.2. Prioriza o nome exato
caminho_exato = Path(ARQUIVO_ALVO)
if caminho_exato.exists():
    caminho_csv = caminho_exato
elif candidatos:
    caminho_csv = candidatos[0]
    print(f"AVISO: '{ARQUIVO_ALVO}' não encontrado. Usando arquivo alternativo: '{caminho_csv.name}'")

# 2.3. Checagem de segurança
if caminho_csv is None:
    raise FileNotFoundError(
        "\n" + "="*70 + "\n"
        "ARQUIVO CSV NÃO ENCONTRADO!\n"
        "="*70 + "\n"
        "Para gerar o gráfico, é necessário o arquivo CSV produzido na Etapa 1.\n\n"
        "Como resolver:\n"
        "  1) Execute primeiro o script da Etapa 1 (análise de mortalidade), OU\n"
        "  2) Faça o upload manual do arquivo 'mortalidade_ac_consolidado.csv'\n"
        "     usando o painel de arquivos do Colab (ícone de pasta à esquerda)\n"
        "     ou com o comando:\n\n"
        "        from google.colab import files\n"
        "        files.upload()\n\n"
        "Depois, execute novamente esta célula.\n"
        + "="*70
    )

print(f"Arquivo CSV carregado: {caminho_csv.resolve()}")

# --- 3. TRATAMENTO E PADRONIZAÇÃO ---
df = pd.read_csv(caminho_csv, encoding="utf-8")

# Conversão numérica robusta
df["ANO"] = pd.to_numeric(df["ANO"], errors="coerce")
df["OBITOS"] = pd.to_numeric(df["OBITOS"], errors="coerce")

# Remoção de linhas inválidas
df = df.dropna(subset=["ANO", "OBITOS", "GRUPO"]).copy()

# Garantir ANO como inteiro (nunca 2010.0)
df["ANO"] = df["ANO"].astype(int)
df["OBITOS"] = df["OBITOS"].astype(int)

# Ordenação cronológica
df = df.sort_values(["ANO", "GRUPO"]).reset_index(drop=True)

# Lista ordenada de anos para o eixo X
anos_ordenados = sorted(df["ANO"].unique().tolist())

print(f"Período: {anos_ordenados[0]} a {anos_ordenados[-1]} | Registros: {len(df)}")

# --- 4. DESIGN E ESTILIZAÇÃO DO GRÁFICO ---
fig = px.line(
    df,
    x="ANO",
    y="OBITOS",
    color="GRUPO",
    markers=True,
    title=(
        "<b>Evolução da Mortalidade por Doenças do Aparelho Circulatório "
        "e Neoplasias — Acre (2010–2023)</b><br>"
        "<sup>Fonte: SIM/DataSUS — Causa básica (CID-10)</sup>"
    ),
    labels={"ANO": "Ano", "OBITOS": "Número de Óbitos", "GRUPO": "Grupo de Causa"},
    template="plotly_white",
    color_discrete_map={
        "Aparelho Circulatório": "#C0392B",   # vermelho
        "Neoplasias": "#2E86C1",              # azul
    },
)

# 4.1. Eixo X com todos os anos explicitamente
fig.update_xaxes(
    tickmode="array",
    tickvals=anos_ordenados,
    ticktext=[str(a) for a in anos_ordenados],
    title_font=dict(size=14, family="Arial"),
    tickfont=dict(size=12, family="Arial"),
    showgrid=True,
    gridcolor="rgba(200,200,200,0.3)",
    dtick=1,
)

# 4.2. Eixo Y formatado
fig.update_yaxes(
    title_font=dict(size=14, family="Arial"),
    tickfont=dict(size=12, family="Arial"),
    showgrid=True,
    gridcolor="rgba(200,200,200,0.3)",
    separatethousands=True,
)

# 4.3. Marcadores e linhas
fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=8, line=dict(width=1, color="white")),
    hovertemplate="<b>%{fullData.name}</b><br>Ano: %{x}<br>Óbitos: %{y:,}<extra></extra>",
)

# 4.4. Layout geral
fig.update_layout(
    title_font=dict(size=16, family="Arial"),
    legend=dict(
        title="Grupo de Causa",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        font=dict(size=12, family="Arial"),
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="rgba(0,0,0,0.15)",
        borderwidth=1,
    ),
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=60, r=40, t=100, b=60),
    height=600,
)

# --- 5. EXIBIÇÃO E DOWNLOAD ---
fig.show()

caminho_html = "mortalidade_grafico_interativo.html"
fig.write_html(caminho_html, include_plotlyjs="cdn")
print(f"Gráfico salvo: {caminho_html}")

# Download automático no Colab
try:
    from google.colab import files
    files.download(caminho_html)
except ImportError:
    print("AVISO: ambiente não é Google Colab. Download manual não disponível.")
except Exception as e:
    print(f"AVISO: falha no download automático: {e}")

Arquivo CSV carregado: /content/mortalidade_ac_consolidado.csv
Período: 2010 a 2023 | Registros: 28


Gráfico salvo: mortalidade_grafico_interativo.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>